In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
import os
os. getcwd()

'/cephfs/scratch/bruening_scratch/lsteuernagel/projects/HypoMap3D/v09_EDA'

In [3]:
hypomap_v09_controls = ad.read_h5ad(filename="/REDACTED-path/HypoMap3D-data/v0.9/9_objects/singlecell/hypomap3d_singlecell_control.h5ad")

In [5]:
hypomap_v09_controls

AnnData object with n_obs × n_vars = 896959 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'sex_per_sample_inferred', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'L1_ID', 'L2_ID', 'L3_ID', 'L4_ID', 'L5_ID', 'L1_descr', 'L2_descr', 'L3_descr', 'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like', 'L4_abc_like', 'L5_abc_like'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    obsm: 'X_embed', 'X_spatial_umap', 'X_umap'

In [6]:
# downsample function (adapted from hypomap v1)
def downsample_balanced_iterative(adata, target_sample_size, predictor_var, stepsize=100, global_seed=1234):
    print("Running downsample_balanced_iterative")

    metadata = adata.obs.copy()
    metadata["Cell_ID"] = metadata.index

    # Remove NAs in predictor_var
    metadata = metadata[~metadata[predictor_var].isna()].copy()

    # Count per group
    group_counts = metadata[predictor_var].value_counts()
    metadata["per_group"] = metadata[predictor_var].map(group_counts)

    # Iterative downsampling
    sizes = np.arange(len(metadata), target_sample_size - 1, -stepsize)
    tmp_samples = metadata.copy()
    np.random.seed(global_seed)

    for size in tqdm(sizes[:-1], desc="Downsampling", unit="step"):
        class_counts = tmp_samples[predictor_var].value_counts()
        n_classes = class_counts.size

        threshold = size / n_classes
        to_downsample = class_counts[class_counts > threshold]

        if not to_downsample.empty:
            min_count = to_downsample.min()
            close_to_downsample = class_counts[class_counts > min_count]
            combined_downsample = pd.Index(np.union1d(to_downsample.index, close_to_downsample.index))

            eligible = tmp_samples[tmp_samples[predictor_var].isin(combined_downsample)]
            leave_out = eligible.sample(n=stepsize, random_state=global_seed).index
            tmp_samples = tmp_samples.drop(index=leave_out)

    adata_subsampled = adata[adata.obs_names.isin(tmp_samples.index)].copy()
    return adata_subsampled


In [7]:
hypomap_v09_controls_downsampled = downsample_balanced_iterative(hypomap_v09_controls, target_sample_size=500000, predictor_var="L2_descr", stepsize=1000)

Running downsample_balanced_iterative


Downsampling: 100%|██████████| 396/396 [05:04<00:00,  1.30step/s]


In [8]:
print("nnz (non-zeros):", hypomap_v09_controls_downsampled.X.data.shape[0])
print("nnz (non-zeros):", hypomap_v09_controls_downsampled.X.data.shape[0] < 2147483647) # required to fit into a stand in-memory Seurat R object

nnz (non-zeros): 1557467426
nnz (non-zeros): True


In [9]:
# Count before downsampling
counts_before = hypomap_v09_controls.obs["L2_descr"].value_counts(dropna=False)
# Count after downsampling
counts_after = hypomap_v09_controls_downsampled.obs["L2_descr"].value_counts(dropna=False)
# Merge into a single DataFrame
comparison_df = pd.DataFrame({
    "Before": counts_before,
    "After": counts_after
}).fillna(0).astype(int)
# Optional: sort by Before column
comparison_df = comparison_df.sort_values(by="Before", ascending=False)

In [10]:
pd.set_option('display.max_rows', None)
print(comparison_df)

                                   Before  After
L2_descr                                        
NN2-1 Oligo_1                       99580  40698
NN2-12 Astro                        90518  36725
NN2-3 OPC                           33101  13413
Gb2-65 SCH Six6.Cdc14a              28206  11334
Gl2-35 VMH Nr5a1.Acvr1c             19806   8000
Gl2-5 MM Foxb1                      18597   7605
Gb2-41 ZI Pax6.Pou6f2               17964   7419
Gl2-33 VMH Fezf1                    17615   7239
Gb2-53 PVpo-VMPO-MPN Hmx2.Isl1      17516   7053
Gl2-30 AHN-RCH-LHA Otp.Fezf1        17383   7014
Gb2-44 MEA-BST Cyp19a1              16964   6797
Gb2-59 AHN-SBPV-PVHd Pdrm12         16928   6889
Gb2-49 DMH Hmx2.Lef1                16860   6792
Gb2-43 DMH-LHA Gsx1                 16191   6592
Gl2-18 MPN-MPO-PVpo Hmx2            14582   6115
Gb2-58 PVR Six3.Sox3                14443   6115
Gb2-39 SI-MPO-LPO Lhx8              14247   6104
Gl2-11 PH-ant-LHA Otp.Bsx           13021   6098
Gb2-52 CP Meis2     

In [17]:
hypomap_v09_controls_downsampled.write_h5ad(filename="/cephfs/scratch/bruening_scratch/lsteuernagel/data/HypoMap3D/v0.9_frozen/hypomap_v09_controls_downsampled500k.h5ad")

In [33]:
## also do a 100k downsampling

In [11]:
hypomap_v09_controls_downsampled_100k = downsample_balanced_iterative(hypomap_v09_controls, target_sample_size=100000, predictor_var="L2_descr", stepsize=1000)

Running downsample_balanced_iterative


Downsampling: 100%|██████████| 796/796 [07:18<00:00,  1.81step/s]


In [12]:
# Count before downsampling
counts_before = hypomap_v09_controls.obs["L2_descr"].value_counts(dropna=False)
# Count after downsampling
counts_after = hypomap_v09_controls_downsampled_100k.obs["L2_descr"].value_counts(dropna=False)
# Merge into a single DataFrame
comparison_df = pd.DataFrame({
    "Before": counts_before,
    "After": counts_after
}).fillna(0).astype(int)
# Optional: sort by Before column
comparison_df = comparison_df.sort_values(by="Before", ascending=False)

In [13]:
pd.set_option('display.max_rows', None)
print(comparison_df)

                                   Before  After
L2_descr                                        
NN2-1 Oligo_1                       99580   3156
NN2-12 Astro                        90518   2760
NN2-3 OPC                           33101   1242
Gb2-65 SCH Six6.Cdc14a              28206   1229
Gl2-35 VMH Nr5a1.Acvr1c             19806   1233
Gl2-5 MM Foxb1                      18597   1236
Gb2-41 ZI Pax6.Pou6f2               17964   1231
Gl2-33 VMH Fezf1                    17615   1233
Gb2-53 PVpo-VMPO-MPN Hmx2.Isl1      17516   1229
Gl2-30 AHN-RCH-LHA Otp.Fezf1        17383   1231
Gb2-44 MEA-BST Cyp19a1              16964   1234
Gb2-59 AHN-SBPV-PVHd Pdrm12         16928   1232
Gb2-49 DMH Hmx2.Lef1                16860   1240
Gb2-43 DMH-LHA Gsx1                 16191   1233
Gl2-18 MPN-MPO-PVpo Hmx2            14582   1226
Gb2-58 PVR Six3.Sox3                14443   1240
Gb2-39 SI-MPO-LPO Lhx8              14247   1227
Gl2-11 PH-ant-LHA Otp.Bsx           13021   1229
Gb2-52 CP Meis2     

In [ ]:
hypomap_v09_controls_downsampled_100k.write_h5ad(filename="/cephfs/scratch/bruening_scratch/lsteuernagel/data/HypoMap3D/v0.9_frozen/hypomap_v09_controls_downsampled100k.h5ad")